# EDA: first-drive RB 5+ rushing yardsRun this after `src/ingest.py` and `src/label.py`. It reproduces the checks in`reports/eda.md` interactively so you can poke at the intermediate frames.The three questions this notebook answers, in the order the product plan asksthem:1. How many usable team-games survive starter resolution?2. What is the base rate of the label, and is this an imbalanced problem?3. What does the distribution of first-drive rushing totals look like?

In [ ]:
import syssys.path.insert(0, "../src")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport ingestfrom config import LABELED_PATH, FEATURES_PATH, YARDS_THRESHOLDpd.set_option("display.width", 200)pd.set_option("display.max_columns", 60)labeled = pd.read_parquet(LABELED_PATH)usable = labeled[labeled.is_ambiguous == 0]len(labeled), len(usable)

## 1. Starter resolutionThe starting RB is the single biggest failure point in this project. Eachteam-game is resolved by taking the RB with the most rush attempts andcross-checking him against Pro-Football-Reference offensive snap share.Disagreements are dropped, not force-labeled.

In [ ]:
print(labeled.starter_method.value_counts(), "\n")print(labeled.loc[labeled.is_ambiguous == 1, "drop_reason"].value_counts())

In [ ]:
# Spot-check a game by hand. Detroit opened 2023 with Montgomery starting# and Gibbs rotating in, which is what the resolver should find.labeled[labeled.game_id == "2023_01_DET_KC"][    ["team", "starter_name", "starter_attempts", "runner_up_attempts",     "starter_snap_pct", "fd_carries", "fd_rush_yards", "label",     "starter_method"]]

## 2. Base rateIf this came back heavily skewed we would need resampling or class weights.It does not.

In [ ]:
base = usable.label.mean()print(f"P(first-drive rush yards >= {YARDS_THRESHOLD}) = {base:.4f}")usable.groupby("season").label.agg(["mean", "size"]).round(4)

### The starter who never touches the ballThis is the most important structural fact in the dataset. The label is acompound event: the back has to be *given* the ball on the opener, and thecarries then have to total 5+ yards. A sixth of all team-games are decided byplay-calling before any rushing play happens.

In [ ]:
zero = (usable.fd_carries == 0).mean()cond = usable.loc[usable.fd_carries > 0, "label"].mean()print(f"zero opening-drive carries: {zero:.1%}")print(f"base rate given >=1 carry:  {cond:.4f}")usable.fd_carries.value_counts().sort_index().head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))carr = usable.groupby("fd_carries").label.agg(["mean", "size"])carr = carr[carr["size"] >= 20]ax.bar(carr.index.astype(int), carr["mean"], color="#4C72B0")ax.set_xlabel("Carries by the starter on the first drive")ax.set_ylabel(f"P(>= {YARDS_THRESHOLD} yards)")ax.set_title("Outcome is driven by opportunity")plt.show()

## 3. Distribution of first-drive rushing yardsThe median sits on the threshold, which is why the base rate lands near 0.5 andwhy a quarter of all outcomes are decided by a yard or two. That caps how sharpany model can be and is the argument for optimizing calibration over accuracy.

In [ ]:
print(usable.fd_rush_yards.describe().round(3))near = usable.fd_rush_yards.between(3, 7).mean()print(f"\nwithin 2 yards of the line: {near:.1%}")fig, ax = plt.subplots(figsize=(7, 4))ax.hist(usable.fd_rush_yards, bins=range(-15, 60), color="#4C72B0")ax.axvline(YARDS_THRESHOLD, color="#C44E52", ls="--",           label=f"{YARDS_THRESHOLD}-yard line")ax.set_xlabel("Starting RB rushing yards on first drive")ax.set_ylabel("Team-games")ax.legend()plt.show()

## 4. Feature correlatesRun `src/features.py` first. Every correlation is small, and the ones that leadare all *opportunity* features rather than talent features. That is thescripted-opener effect the product plan warned about, visible in the data.

In [ ]:
import features as Ffeats = pd.read_parquet(FEATURES_PATH)fu = feats[feats.is_ambiguous == 0]cols = [c for c in F.feature_columns(fu) if fu[c].notna().sum() > 100](fu[cols + ["label"]].corr()["label"].drop("label")   .sort_values(key=np.abs, ascending=False).head(15).round(4))

## 5. Leakage sanity checkEvery rolling feature must be built from strictly prior games. The formal proofis in `tests/test_leakage.py`, which rebuilds features with the future deletedand asserts nothing changes. Here is the quick visual version: a player's firstcareer game must have no trailing features at all.

In [ ]:
debut = fu.sort_values(["season", "week"]).groupby("starter_id").head(1)trailing = [c for c in cols if c.endswith("_t5")]share_null = debut[trailing].isna().mean().sort_values(ascending=False)print("share of debut rows with a null trailing feature:")print(share_null.head(8).round(3))